# AgentBench-Gov: Dataset Exploration

This notebook provides a comprehensive exploration of the AgentBench-Gov dataset:
500 governance evaluation tasks across 5 dimensions and 3 difficulty levels.

**Contents:**
1. Dataset overview and statistics
2. Dimension and sub-category distribution
3. Difficulty distribution analysis
4. Task structure deep-dive
5. Expected element analysis
6. Regulatory framework coverage
7. Tag frequency analysis

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from pathlib import Path

# Style
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Load dataset
ROOT = Path('..')
with open(ROOT / 'datasets' / 'governance_tasks.json') as f:
    data = json.load(f)

tasks = data['tasks']
df = pd.DataFrame(tasks)
print(f'Loaded {len(df)} tasks')
df.head(2)

## 1. Dataset Overview

In [ ]:
print('=== DATASET SUMMARY ===')
print(f'Total tasks:      {len(df)}')
print(f'Dimensions:       {df["dimension"].nunique()}')
print(f'Sub-categories:   {df["sub_category"].nunique()}')
print(f'Difficulty levels: {df["difficulty"].nunique()}')
print(f'Frameworks:       {df["regulatory_framework"].nunique() if "regulatory_framework" in df.columns else "N/A"}')
print()
print('Tasks per dimension:')
print(df['dimension'].value_counts().to_string())
print()
print('Tasks per difficulty:')
print(df['difficulty'].value_counts().to_string())

## 2. Dimension and Sub-Category Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dimension distribution
dim_counts = df['dimension'].value_counts()
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']
axes[0].bar(dim_counts.index, dim_counts.values, color=colors)
axes[0].set_title('Tasks per Dimension', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Number of Tasks')
axes[0].tick_params(axis='x', rotation=15)
for i, (x, y) in enumerate(zip(dim_counts.index, dim_counts.values)):
    axes[0].text(i, y + 1, str(y), ha='center', fontweight='bold')

# Sub-category distribution
sub_counts = df['sub_category'].value_counts()
axes[1].barh(sub_counts.index, sub_counts.values, color='#2196F3', alpha=0.8)
axes[1].set_title('Tasks per Sub-Category', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Number of Tasks')
for i, (y, x) in enumerate(zip(sub_counts.index, sub_counts.values)):
    axes[1].text(x + 0.5, i, str(x), va='center')

plt.tight_layout()
plt.savefig('../figures/notebook_fig_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Difficulty Distribution

In [ ]:
# Difficulty by dimension
cross = pd.crosstab(df['dimension'], df['difficulty'])
cross = cross.reindex(columns=['easy', 'medium', 'hard'])

fig, ax = plt.subplots(figsize=(11, 5))
cross.plot(kind='bar', ax=ax, color=['#4CAF50', '#FF9800', '#F44336'])
ax.set_title('Difficulty Distribution by Dimension', fontweight='bold', fontsize=13)
ax.set_xlabel('Dimension')
ax.set_ylabel('Number of Tasks')
ax.tick_params(axis='x', rotation=15)
ax.legend(title='Difficulty')
plt.tight_layout()
plt.show()

print('\nDifficulty cross-tabulation:')
print(cross.to_string())
print(f'\nOverall difficulty breakdown:')
print(df['difficulty'].value_counts(normalize=True).mul(100).round(1).to_string())

## 4. Task Structure Analysis

In [ ]:
# Scenario and question length analysis
df['scenario_words'] = df['scenario'].apply(lambda x: len(str(x).split()))
df['question_words'] = df['question'].apply(lambda x: len(str(x).split()))
df['n_elements'] = df['expected_elements'].apply(len)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df['scenario_words'], bins=20, color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].set_title('Scenario Length (words)', fontweight='bold')
axes[0].set_xlabel('Word count')
axes[0].axvline(df['scenario_words'].mean(), color='red', linestyle='--', label=f'Mean: {df["scenario_words"].mean():.0f}')
axes[0].legend()

axes[1].hist(df['question_words'], bins=15, color='#4CAF50', alpha=0.8, edgecolor='white')
axes[1].set_title('Question Length (words)', fontweight='bold')
axes[1].set_xlabel('Word count')
axes[1].axvline(df['question_words'].mean(), color='red', linestyle='--', label=f'Mean: {df["question_words"].mean():.0f}')
axes[1].legend()

axes[2].hist(df['n_elements'], bins=range(1, 12), color='#FF9800', alpha=0.8, edgecolor='white')
axes[2].set_title('Expected Elements per Task', fontweight='bold')
axes[2].set_xlabel('Number of elements')
axes[2].axvline(df['n_elements'].mean(), color='red', linestyle='--', label=f'Mean: {df["n_elements"].mean():.1f}')
axes[2].legend()

plt.tight_layout()
plt.show()

print('Task structure statistics:')
print(df[['scenario_words', 'question_words', 'n_elements']].describe().round(1).to_string())

## 5. Expected Element Length Analysis

In [ ]:
# Length distribution of expected elements
all_elements = [e for task_elements in df['expected_elements'] for e in task_elements]
element_lengths = [len(e.split()) for e in all_elements]

print(f'Total expected elements in dataset: {len(all_elements)}')
print(f'Mean element length: {np.mean(element_lengths):.1f} words')
print(f'Median element length: {np.median(element_lengths):.1f} words')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(element_lengths, bins=20, color='#9C27B0', alpha=0.8, edgecolor='white')
ax.set_title('Expected Element Length Distribution', fontweight='bold', fontsize=13)
ax.set_xlabel('Words per element')
ax.set_ylabel('Frequency')
ax.axvline(np.mean(element_lengths), color='red', linestyle='--',
           label=f'Mean: {np.mean(element_lengths):.1f}')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Regulatory Framework Coverage

In [ ]:
if 'regulatory_framework' in df.columns:
    fw_counts = df['regulatory_framework'].value_counts()
    fig, ax = plt.subplots(figsize=(9, 4))
    wedges, texts, autotexts = ax.pie(
        fw_counts.values,
        labels=fw_counts.index,
        autopct='%1.1f%%',
        colors=plt.cm.Set3.colors[:len(fw_counts)]
    )
    ax.set_title('Regulatory Framework Coverage', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.show()
    print(fw_counts.to_string())
else:
    # Fall back to sub_category as proxy for regulatory framework
    sub_map = {
        'gdpr': 'GDPR', 'ai_act': 'EU AI Act', 'hipaa': 'HIPAA',
        'financial': 'Financial (SOX/MiFID II/ECOA)', 'explainability': 'IEEE 7000/EU HLEG',
        'audit': 'NIST AI RMF', 'risk': 'NIST AI RMF / EU AI Act', 'consistency': 'ISO 9001'
    }
    df['framework_mapped'] = df['sub_category'].map(sub_map).fillna(df['sub_category'])
    fw_counts = df['framework_mapped'].value_counts()
    print('Regulatory framework mapping (by sub-category):')
    print(fw_counts.to_string())

## 7. Summary Table

In [ ]:
summary = df.groupby(['dimension', 'difficulty']).size().unstack(fill_value=0)
summary = summary.reindex(columns=['easy', 'medium', 'hard'])
summary['total'] = summary.sum(axis=1)
print('Dataset Summary Table:')
print(summary.to_string())
print(f'\nGrand total: {summary["total"].sum()}')